# Processamento de Língua Natural - EP (Testes)
Baseline: Regressão Logística com contagem TF-IDF

## Bibliotecas
Sklearn, Pandas

In [64]:
import re
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline

## Importação do Conjunto de Treino
O Pandas importa ambos os conjuntos de dados (treino e teste) como strings, para evitar erros de tipagem no momento da tokenização.

In [65]:
dados_treino = pd.read_excel("dados/train.xlsx", converters={'resp_text':str,'clarity':str})

X_treino, Y_treino = dados_treino["resp_text"].tolist(), dados_treino["clarity"].tolist()

In [66]:
dados_treino

,resp_text,clarity
0,"Prezado(a) Senhor(a), Esclarecemos que o Se...",c5
1,"Prezada cidadã, As informações sobre óbitos ...",c1
2,"Prezado Senhor Julio, A Ouvidoria-Geral da P...",c1
3,"Prezado(a) Senhor(a), Esclarecemos que o Se...",c234
4,"Senhor, O Serviço de Informações ao Cidadão d...",c234
...,...,...
20087,Prezado Sr. Luciano Wolff A sua solicitação ...,c1
20088,"PREZADO CIDADãO, A COORDENAçãO DO PROGRAMA FA...",c1
20089,"Prezado, Informamos que o Ministério da Agr...",c1
20090,"Prezado Senhor, Encaminhamos em anexo respos...",c1


## Pré-processamento e Tokenização

In [67]:
def preprocess_text(text):
    text = str(text).lower()

    # Normaliza elementos muito específicos, preservando a informação de que eles existem.
    text = re.sub(r'https?://\S+|www\.\S+', ' url ', text)
    text = re.sub(r'\b[\w\.-]+@[\w\.-]+\.\w+\b', ' email ', text)
    text = re.sub(r'\b\d+(?:[\.,]\d+)?\b', ' num ', text)

    # Normaliza espaços e quebras de linha.
    text = re.sub(r'\s+', ' ', text).strip()

    return text


vect = TfidfVectorizer(
    preprocessor=preprocess_text,
    analyzer='word',
    ngram_range=(1, 2),
    min_df=10,
    max_df=0.95,
    max_features=20000,
    sublinear_tf=True
)

## Classificador (Regressão Logística)

In [68]:
clf = LogisticRegression(
    C=0.5,
    class_weight='balanced',
    max_iter=150,
    solver='saga',
    tol=1e-3,
    random_state=42
)

## Validação Cruzada

In [69]:
pipeline = Pipeline([
    ('tfidf', vect),
    ('clf', clf)
])

# Durante a experimentação usamos 5 folds para reduzir o tempo de execução.
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

resultados = cross_validate(
    pipeline,
    X_treino,
    Y_treino,
    scoring='accuracy',
    cv=cv,
    return_train_score=True,
    n_jobs=-1
)

acuracia_validacao = resultados['test_score'].mean()
acuracia_treino_cv = resultados['train_score'].mean()
gap = acuracia_treino_cv - acuracia_validacao

print(f"Acurácia Média da Validação Cruzada: {acuracia_validacao:.2%}")
print(f"Acurácia Média de Treino nos Folds: {acuracia_treino_cv:.2%}")
print(f"Gap Treino-Validação: {gap:.2%}")

Acurácia Média da Validação Cruzada: 45.74%
Acurácia Média de Treino nos Folds: 64.86%
Gap Treino-Validação: 19.12%


## Treino do Modelo

In [70]:
pipeline.fit(X_treino, Y_treino)
print(f"Acurácia do Treino (modelo final): {pipeline.score(X_treino, Y_treino):.2%}")

Acurácia do Treino (modelo final): 63.60%


## Execução do Teste e Salvamento do Resultado

In [71]:
# Importação dos dados de teste
dados_teste = pd.read_excel("dados/test1.xlsx", converters={'resp_text': str, 'clarity': str})
X_teste = dados_teste["resp_text"].tolist()

# Aplicação do modelo no teste
Y_previsto = pipeline.predict(X_teste)

In [72]:
teste_rotulado = pd.DataFrame({
    'resp_text': X_teste,
    'clarity': Y_previsto
})

teste_rotulado.to_excel("teste_rotulado.xlsx", index=False)